# Loading Pretrained GPT-2 Weights

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import tiktoken
from minigpt.pretrained import load_gpt2
from minigpt.generate import generate

## 1. Load Pretrained Model

In [ ]:
model = load_gpt2('gpt2')
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')
print(f'Weight tying: {model.tok_emb.weight is model.lm_head.weight}')

## 2. Generate Text

In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')

def gen(prompt, max_new_tokens=100):
    input_ids = torch.tensor([tokenizer.encode(prompt)])
    with torch.no_grad():
        output_ids = generate(model, input_ids, max_new_tokens=max_new_tokens, context_length=1024)
    return tokenizer.decode(output_ids[0].tolist())

In [ ]:
prompts = [
    'To be or not to be',
    'The meaning of life is',
    'In the beginning, there was',
    'Machine learning is a field of',
    'Hello, my name is',
]

for prompt in prompts:
    print(f'--- {prompt} ---')
    print(gen(prompt))
    print()

## 3. Verify Against HuggingFace

Proof that our from-scratch model produces identical outputs.

In [ ]:
from transformers import GPT2LMHeadModel

hf_model = GPT2LMHeadModel.from_pretrained('gpt2', cache_dir='../data/hf_cache')
hf_model.eval()

test_input = torch.tensor([[15496, 11, 616, 1438, 318]])  # "Hello, my name is"

with torch.no_grad():
    our_logits = model(test_input)
    hf_logits = hf_model(test_input).logits

max_diff = (our_logits - hf_logits).abs().max().item()
print(f'Max logit difference: {max_diff:.2e}')
print(f'Match: {torch.allclose(our_logits, hf_logits, atol=1e-4)}')